In [1]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 3.5 MB/s  0:00:05m0:00:0100:01


In [1]:
import mysql.connector

connection=mysql.connector.connect(
    host="localhost",
    user="root",
    password="12345678",
    database="traffic_crash_analysis"
)

cursor=connection.cursor(buffered=True)
cursor

In [2]:
query="select * from crash_data limit 10"
cursor.execute(query)
for data in cursor:
    print(data)

('000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9eee259ec4975d55185c0c51bea918a1c5fb4520cec9c7a50ac442c1671a2d41eaf05442c3b34b9af5b', '01/14/2025 12:25:00 PM', 30, 'TRAFFIC SIGNAL', 'FUNCTIONING PROPERLY', 'SNOW', 'DAYLIGHT', 'SIDESWIPE SAME DIRECTION', 'DIVIDED - W/MEDIAN (NOT RAISED)', 'STRAIGHT AND LEVEL', 'SNOW OR SLUSH', 'NO DEFECTS', 'ON SCENE', 'NO INJURY / DRIVE AWAY', '$501 - $1,500', '01/14/2025 12:38:00 PM', 'IMPROPER TURNING/NO SIGNAL', 'IMPROPER OVERTAKING/PASSING', 6352, 'N', 'SHERIDAN RD', 2433.0, 2, 'NO INDICATION OF INJURY', 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 12, 3, 1, 41.9978077276, -87.6557704947, 'POINT (-87.655770494712 41.997807727633)', '2025-01-14 12:25:00', 2025)
('027b0b4c21460d3441fd83929abb9673c6fc0c7d5756753e2e5ce89a39c6f07339443c29066246d583d0f2ec5d6c427e010229e240813616aee55cd8a01c5108', '05/23/2025 09:30:00 AM', 30, 'STOP SIGN/FLASHER', 'UNKNOWN', 'UNKNOWN', 'DAYLIGHT', 'TURNING', 'DIVIDED - W/MEDIAN (NOT RAISED)', 'STRAIGHT AND LEVEL', 'UNKNOWN', 'UNKNOW

In [3]:
query="select count(*) from crash_data limit 10"
cursor.execute(query)
for data in cursor:
    print(data)

(660934,)


1. Find the top 5 most dangerous combinations of weather and crash type based on total crashes. 

In [4]:
query="""select weather_condition, crash_type, count(*) as total_crashes from crash_data 
group by weather_condition, crash_type order by total_crashes desc limit 5;"""
cursor.execute(query)
for data in cursor:
    print(data)


('CLEAR', 'NO INJURY / DRIVE AWAY', 360238)
('CLEAR', 'INJURY AND / OR TOW DUE TO CRASH', 155302)
('UNKNOWN', 'NO INJURY / DRIVE AWAY', 41442)
('RAIN', 'NO INJURY / DRIVE AWAY', 31560)
('RAIN', 'INJURY AND / OR TOW DUE TO CRASH', 19647)


2. Identify the top 10 streets with the highest number of injury crashes.

In [6]:
query="select street_name,count(*) as Injuries from crash_data where crash_type like 'Injury%' group by street_name order by injuries desc limit 10;"
cursor.execute(query)
for data in cursor:
    print(data)


('WESTERN AVE', 5334)
('PULASKI RD', 4899)
('CICERO AVE', 4515)
('ASHLAND AVE', 4368)
('HALSTED ST', 4124)
('KEDZIE AVE', 3539)
('LAKE SHORE DR SB', 2449)
('LAKE SHORE DR NB', 2314)
('STONY ISLAND AVE', 2252)
('STATE ST', 2231)


3. Find the percentage of crashes that resulted in injuries for each crash type. 

In [7]:
query="select crash_type, round((sum(injuries_total)/count(*))*100,2) as percentage from crash_data group by crash_type having crash_type like 'INJURY%';"
cursor.execute(query)
for data in cursor:
    print(data)


('INJURY AND / OR TOW DUE TO CRASH', 73.51)


4. Determine the peak crash hour for each month.

In [ ]:
query="""select crash_month, crash_hour, total_crashes from(
select crash_month, crash_hour, count(*) as total_crashes , 
dense_rank() over(partition by crash_month order by count(*) desc) as ranked from crash_data
group by crash_month, crash_hour)as tab
where ranked=1;"""
cursor.execute(query)
for data in cursor:
    print(data)

(1, 15, 4408)
(2, 15, 4525)
(3, 15, 4610)
(4, 15, 3884)
(5, 15, 4727)
(6, 16, 4552)
(7, 16, 4299)
(8, 16, 4504)
(9, 15, 4597)
(10, 15, 4644)
(11, 17, 3782)
(12, 17, 4117)


5. Find the top 5 primary causes of crashes during night time (CRASH_HOUR ≥ 18). 

In [ ]:
query="select prim_contributory_cause,count(*) as total_crash from crash_data where crash_hour>=18 group by prim_contributory_cause order by total_crash desc limit 5;"
cursor.execute(query)
for data in cursor:
    print(data)


('UNABLE TO DETERMINE', 20)
('DISREGARDING TRAFFIC SIGNALS', 23)
('IMPROPER OVERTAKING/PASSING', 22)
('UNABLE TO DETERMINE', 20)
('IMPROPER OVERTAKING/PASSING', 22)


6. Compare average number of injuries in daylight vs darkness conditions. 

In [8]:
query="""select lighting_condition,avg(injuries_total) as average_injury from crash_data group by lighting_condition having lighting_condition like "daylight%" or lighting_condition like "darkness%";"""
cursor.execute(query)
for data in cursor:
    print(data)

('DAYLIGHT', 0.20705380553816333)
('DARKNESS, LIGHTED ROAD', 0.2858614976359497)
('DARKNESS', 0.21685270895725564)


7. Find which traffic control device type has the highest average injuries per crash.

In [ ]:
query="""select traffic_control_device,avg(injuries_total) as average_injury from crash_data group by traffic_control_device order by average_injury desc limit 1;"""
cursor.execute(query)
for data in cursor:
    print(data)

('BICYCLE CROSSING SIGN', 0.6551724137931034)


8. Identify the top 5 locations (latitude/longitude) with the highest crash frequency. 

In [9]:
query="""select location,count(*) as frequency from crash_data group by location order by frequency desc limit 5;"""
cursor.execute(query)
for data in cursor:
    print(data)

('POINT (-87.905309125103 41.976201139024)', 1247)
('POINT (-87.619928173678 41.900958919109)', 661)
('POINT (-87.580147768689 41.791420282098)', 473)
('POINT (-87.585971992965 41.751460603167)', 469)
('POINT (-87.585275565077 41.722257273006)', 353)


9. Find the top 5 streets with the highest injury rate, considering only streets with more than 100 crashes. 

In [11]:
query="""select street_name,count(*) as crashes, round(sum(injuries_total)/count(*),2) as injury_rate from crash_data group by street_name having crashes>100 order by injury_rate desc limit 5;"""
cursor.execute(query)
for data in cursor:
    print(data)

('MARQUETTE DR', 271, 0.45)
('FIFTH AVE', 299, 0.43)
('CORCORAN PL', 122, 0.4)
('SOUTH CHICAGO AVE', 1710, 0.39)
('DOUGLAS BLVD', 390, 0.39)


10 . For each year, identify the most common crash type. 

In [10]:
query="""select year,crash_type,total_crash from(
select year,crash_type, count(*) as total_crash , dense_rank() over (partition by year order by count(*) desc) as ranked from crash_data group by year,crash_type) as tab
where ranked=1; """
cursor.execute(query)
for data in cursor:
    print(data)

(2020, 'NO INJURY / DRIVE AWAY', 62706)
(2021, 'NO INJURY / DRIVE AWAY', 74661)
(2022, 'NO INJURY / DRIVE AWAY', 75897)
(2023, 'NO INJURY / DRIVE AWAY', 78034)
(2024, 'NO INJURY / DRIVE AWAY', 78560)
(2025, 'NO INJURY / DRIVE AWAY', 77388)
(2026, 'NO INJURY / DRIVE AWAY', 18547)


11. Find the day of the week with the highest average crashes per hour.

In [ ]:
query="""select crash_day_of_week, round(avg(total_crash),2) as average from
(select crash_day_of_week,crash_hour,count(*) as total_crash from crash_data group by crash_day_of_week,crash_hour order by crash_day_of_week) as tab
group by crash_day_of_week
order by average desc limit 1;"""
cursor.execute(query)
for data in cursor:
    print(data)

(6, Decimal('4455.17'))


12. Identify high-risk time slots:
Group hours into buckets (Morning, Afternoon, Evening, Night)
Find which bucket has the highest injury crashes


In [12]:
query="""select case when crash_hour between 4 and 11 then "Morning"
when crash_hour between 12 and 16 then "Afternoon"
when crash_hour between 17 and 20 then "Evening" else "Night" end as time_bucket,
sum(injuries_total) as injuries
from crash_data
group by time_bucket
order by injuries desc limit 1;"""
cursor.execute(query)
for data in cursor:
    print(data)

('Afternoon', 44239.0)


13. Find the top 3 contributing causes for each crash type.(Use window functions like ROW_NUMBER() or RANK())

In [15]:
query="""select crash_type,prim_contributory_cause from
(select crash_type, prim_contributory_cause, count(*) as total , 
dense_rank() over(partition by crash_type order by count(*) desc) as ranked from crash_data 
group by crash_type,prim_contributory_cause) as tab
where ranked<=3;"""
cursor.execute(query)
for data in cursor:
    print(data)

('INJURY AND / OR TOW DUE TO CRASH', 'UNABLE TO DETERMINE')
('INJURY AND / OR TOW DUE TO CRASH', 'FAILING TO YIELD RIGHT-OF-WAY')
('INJURY AND / OR TOW DUE TO CRASH', 'FAILING TO REDUCE SPEED TO AVOID CRASH')
('NO INJURY / DRIVE AWAY', 'UNABLE TO DETERMINE')
('NO INJURY / DRIVE AWAY', 'FOLLOWING TOO CLOSELY')
('NO INJURY / DRIVE AWAY', 'FAILING TO YIELD RIGHT-OF-WAY')


14. Calculate the year-over-year growth rate of crashes.

In [16]:
query="""select year, count(*) as current_total_crashes , lag(count(*)) over(order by year) as previous_year_crashes,
round((count(*)- lag(count(*)) over(order by year)) *100 / lag(count(*)) over(order by year),2) as growth_rate
from crash_data
group by year;"""
cursor.execute(query)
for data in cursor:
    print(data)

(2020, 91509, None, None)
(2021, 107937, 91509, Decimal('17.95'))
(2022, 107462, 107937, Decimal('-0.44'))
(2023, 109714, 107462, Decimal('2.10'))
(2024, 110853, 109714, Decimal('1.04'))
(2025, 107965, 110853, Decimal('-2.61'))
(2026, 25494, 107965, Decimal('-76.39'))


15. Identify hotspot zones: Group nearby locations (round latitude & longitude to 2 decimal places),Find top 10 zones with highest crashes


In [17]:
query="""select round(latitude,2) as latitude, round(longitude,2) as longitude, count(*) as total
from crash_data 
group by round(latitude,2),round(longitude,2)
order by total desc limit 10;"""
cursor.execute(query)
for data in cursor:
    print(data)

(41.89, -87.63, 7633)
(41.88, -87.63, 5552)
(41.89, -87.62, 5550)
(41.9, -87.62, 5049)
(41.88, -87.62, 4272)
(41.88, -87.64, 3951)
(41.9, -87.63, 3935)
(41.99, -87.66, 3392)
(41.89, -87.64, 3158)
(41.87, -87.64, 3111)
